- VLM 모델 학습을 위해 COCO 데이터셋 처럼 하나의 이미지 당 5개의 문장 라벨링
- Gemini 2.5 Flash API를 활용한 라벨링

In [4]:
import google.generativeai as genai
import json
import os
from PIL import Image
import shutil
from dotenv import load_dotenv
import requests
import base64

c:\Users\ASUS\anaconda3\envs\ds_study\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.8.20). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


In [11]:

# 원본 이미지 경로: 이전 프로젝트 경로에 있음
IMG_SRC = r"C:\Users\ASUS\Desktop\제로베이스\딥러닝 프로젝트\낙상사고 위험동작 영상-센서 쌍 데이터_병원,후면낙상\3.개방데이터\1.데이터\Training\01.원천데이터\TS\이미지"

# 원본 라벨 경로: 현재 프로젝트 경로에 있음
PROJECT = r"C:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM"
LBL_SRC = os.path.join(PROJECT, "labels", "이미지")

# 임시 폴더
TMP_IMG = os.path.join(PROJECT, "tmp_test", "images")
TMP_LBL = os.path.join(PROJECT, "tmp_test", "labels_original")
TMP_OUT = os.path.join(PROJECT, "tmp_test", "test_labels")

# 기존 정리 후 재생성
if os.path.exists(os.path.join(PROJECT, "tmp_test")):
    shutil.rmtree(os.path.join(PROJECT, "tmp_test"))

os.makedirs(TMP_IMG, exist_ok=True)
os.makedirs(TMP_LBL, exist_ok=True)
os.makedirs(TMP_OUT, exist_ok=True)

# ★ 1개 시나리오의 이미지 10장 전부 복사
scenario = "00131_H_A_SY_C3"

img_folder = os.path.join(IMG_SRC, "Y", "SY", scenario)
lbl_folder = os.path.join(LBL_SRC, "Y", "SY", scenario)
for f in sorted(os.listdir(img_folder)):
    if not f.lower().endswith('.jpg'):
        continue
    
    # 이미지 복사
    shutil.copy2(os.path.join(img_folder, f), os.path.join(TMP_IMG, f))
    
    # 대응 라벨 복사
    json_name = f.replace(".jpg", ".json").replace(".JPG", ".json")
    lbl_path = os.path.join(lbl_folder, json_name)
    if os.path.exists(lbl_path):
        shutil.copy2(lbl_path, os.path.join(TMP_LBL, json_name))
print(f"시나리오: {scenario}")
print(f"이미지 {len(os.listdir(TMP_IMG))}장: {sorted(os.listdir(TMP_IMG))}")
print(f"라벨 {len(os.listdir(TMP_LBL))}개: {sorted(os.listdir(TMP_LBL))}")

시나리오: 00131_H_A_SY_C3
이미지 10장: ['00131_H_A_SY_C3_I001.JPG', '00131_H_A_SY_C3_I002.JPG', '00131_H_A_SY_C3_I003.JPG', '00131_H_A_SY_C3_I004.JPG', '00131_H_A_SY_C3_I005.JPG', '00131_H_A_SY_C3_I006.JPG', '00131_H_A_SY_C3_I007.JPG', '00131_H_A_SY_C3_I008.JPG', '00131_H_A_SY_C3_I009.JPG', '00131_H_A_SY_C3_I010.JPG']
라벨 10개: ['00131_H_A_SY_C3_I001.json', '00131_H_A_SY_C3_I002.json', '00131_H_A_SY_C3_I003.json', '00131_H_A_SY_C3_I004.json', '00131_H_A_SY_C3_I005.json', '00131_H_A_SY_C3_I006.json', '00131_H_A_SY_C3_I007.json', '00131_H_A_SY_C3_I008.json', '00131_H_A_SY_C3_I009.json', '00131_H_A_SY_C3_I010.json']


In [12]:
# .env 파일 로드
load_dotenv("API_KEY_to_Labelling.env")
API_KEY = os.getenv("GOOGLE_API_KEY")
URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={API_KEY}"

In [32]:
PROMPT = """당신은 병원 환자 안전 모니터링 전문가입니다.
이 이미지를 분석하여 환자의 현재 상태를 보고하세요.

## 판단 기준 (우선순위 순)

1. [fall] 환자의 몸(무릎, 엉덩이, 상체 등)이 바닥에 접촉 → "낙상 발생"
   - 넘어져서 바닥에 쓰러져 있거나, 바닥에 주저앉은 경우

2. [bed_exit] 환자가 침대에서 벗어나려는 '동작'이 관찰됨 → "침대 이탈"
   - 침대에서 내려오는 동작
   - 침대에서 내려와 서있는 상태
   ※ 단, 침대 위에서 편안히 앉거나 누워있는 자세는 이탈이 아님

3. [moving] 환자가 침대 밖에서 서서 걷거나 이동 중 → "이동 중"
   - 보행 보조기 사용 여부 무관

4. [resting] 환자가 침대 안에서 안정된 자세로 있음 → "휴식 중"
   - 침대에 누워있는 상태
   - 침대 위에 편안히 앉아있는 상태 (이탈 동작 없음)
   - 이불을 덮고 있거나, 베개에 기대어 있는 경우

## 핵심 구분 기준
- "침대에 앉아있다" → 침대에서 나오는 동작이면 bed_exit, 편안히 앉아있거나 누워있으면 resting
- "바닥에 있다" → 쓰러져있으면 fall, 서있으면 moving

## 출력 형식 (반드시 JSON으로)
{
  "status": "fall | bed_exit | moving | resting",
  "captions": [
    "한국어 상태 설명 1 (10~20단어, 의료 보고서 문체)",
    "한국어 상태 설명 2 (같은 상황을 다른 표현으로)",
    "한국어 상태 설명 3 (자세/위치 중심 묘사)",
    "한국어 상태 설명 4 (위험도 중심 묘사)",
    "한국어 상태 설명 5 (간결한 한 줄 보고)"
  ]
}

JSON만 출력하세요."""

In [18]:
def call_gemini(image_path, prompt):
    """Gemini 2.5 Flash API 호출 (REST)"""
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode("utf-8")
    
    payload = {
        "contents": [{
            "parts": [
                {"text": prompt},
                {"inline_data": {"mime_type": "image/jpeg", "data": img_b64}}
            ]
        }]
    }
    
    response = requests.post(URL, json=payload)
    result = response.json()
    
    # ★ 디버그: API 응답 전체 출력
    if "candidates" not in result:
        print(f"  [DEBUG] API 응답: {json.dumps(result, indent=2, ensure_ascii=False)[:500]}")
        return None
    
    text = result["candidates"][0]["content"]["parts"][0]["text"].strip()
    
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    
    return json.loads(text)

In [19]:
test_result = call_gemini(
    os.path.join(TMP_IMG, "00131_H_A_SY_C3_I002.JPG"),
    PROMPT  # ← "이 이미지에 무엇이 보이나요?" 대신 PROMPT 사용!
)
print(test_result)

{'status': 'bed_exit', 'captions': ['환자가 침대 가장자리에 앉아 발을 내리는 동작으로, 침대 이탈을 시도하는 모습이 명확히 관찰됩니다.', '환자가 침대 밖으로 내려오기 위해 움직이고 있으며, 현재 침상에서 발을 내리는 중입니다.', '환자는 침대 끝에 걸터앉아 있으며, 발이 침대 바깥쪽으로 향한 채 지면에 닿기 직전의 자세를 취하고 있습니다.', '환자가 침대 이탈을 시도 중이며, 보호자의 즉각적인 개입이 없을 시 낙상 위험이 높은 상황입니다.', '환자가 침대에서 이탈하려는 동작 중.']}


In [20]:
PROJECT = r"C:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM"
TMP_IMG = os.path.join(PROJECT, "tmp_test", "images")
TMP_LBL = os.path.join(PROJECT, "tmp_test", "labels_original")
TMP_OUT = os.path.join(PROJECT, "tmp_test", "test_labels")

for img_name in sorted(os.listdir(TMP_IMG)):
    if not img_name.lower().endswith('.jpg'):
        continue
    
    json_name = img_name.replace(".jpg", ".json").replace(".JPG", ".json")
    label_path = os.path.join(TMP_LBL, json_name)
    
    # 1) 기존 JSON 읽기
    if not os.path.exists(label_path):
        print(f"[SKIP] 라벨 없음: {json_name}")
        continue
    
    with open(label_path, 'r', encoding='utf-8') as f:
        original_data = json.load(f)
    
    print(f"\n처리 중: {img_name}")
    
    # 2) Gemini API 호출
    try:
        vlm_result = call_gemini(os.path.join(TMP_IMG, img_name), PROMPT)
        
        # 3) 기존 JSON에 vlm_labels 병합
        original_data["vlm_labels"] = {
            "status": vlm_result["status"],
            "captions": vlm_result["captions"]
        }
        
        # 4) test_labels에 저장
        save_path = os.path.join(TMP_OUT, json_name)
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(original_data, f, ensure_ascii=False, indent=2)
        
        print(f"  상태: {vlm_result['status']}")
        for i, cap in enumerate(vlm_result['captions'], 1):
            print(f"  캡션{i}: {cap}")
            
    except Exception as e:
        print(f"  [ERROR] {e}")

print(f"\n완료! 결과: {sorted(os.listdir(TMP_OUT))}")


처리 중: 00131_H_A_SY_C3_I001.JPG
  상태: resting
  캡션1: 환자는 현재 침대 위에 편안하게 앉아 휴식 중이며, 특이사항 없이 안정적인 상태를 유지하고 있습니다.
  캡션2: 현재 환자는 침상에 앉아 정적인 자세로 쉬고 있으며, 침대 이탈 징후는 관찰되지 않습니다.
  캡션3: 환자는 침대 중앙에 다리를 모아 앉아있으며, 상체는 곧게 유지된 안정적인 자세로 관찰됩니다.
  캡션4: 낙상 또는 침대 이탈과 관련된 즉각적인 위험은 현재 없으며, 환자는 안전하게 침상에 머무르고 있습니다.
  캡션5: 환자는 침대 위에서 안정적으로 휴식 중입니다.

처리 중: 00131_H_A_SY_C3_I002.JPG
  상태: bed_exit
  캡션1: 환자가 침대 가장자리에 앉아 양발을 바닥으로 내리는 동작을 보이며 침대 이탈을 시도하고 있습니다.
  캡션2: 현재 환자는 침대 밖으로 나오기 위해 몸을 움직이며 다리를 침대 아래로 내린 상태가 관찰됩니다.
  캡션3: 환자가 침대 끝에 걸터앉아 두 발을 공중에 둔 채 침대에서 내려오려는 자세를 취하고 있습니다.
  캡션4: 침대에서 이탈하려는 환자의 모습이 포착되어 낙상 위험이 있으므로 즉각적인 주의와 개입이 필요합니다.
  캡션5: 환자가 침대에서 이탈하려는 동작이 감지되었습니다.

처리 중: 00131_H_A_SY_C3_I003.JPG
  상태: bed_exit
  캡션1: 환자가 침대 가장자리에 앉아 목발을 잡고 있으며, 침대에서 벗어나려는 동작을 준비하는 상태가 관찰됩니다.
  캡션2: 현재 환자는 침대 끝에 걸터앉아 하지를 내리고 있으며, 목발을 짚어 침대 이탈을 시도하는 상황입니다.
  캡션3: 환자가 침상 가장자리에 앉아 발을 바닥 쪽으로 내린 채 목발을 잡고 있어 침상 이탈 준비 자세입니다.
  캡션4: 침대 이탈을 시도하는 환자의 움직임이 감지되어 낙상 위험이 증가하므로 즉각적인 확인과 지원이 필요합니다.
  캡션5: 환자가 침대 끝에 앉아 목발을 잡고 침대 밖으로 내려오려는 

In [21]:
for f in sorted(os.listdir(TMP_OUT)):
    with open(os.path.join(TMP_OUT, f), 'r', encoding='utf-8') as fp:
        data = json.load(fp)
    
    print(f"\n{'='*50}")
    print(f"파일: {f}")
    print(f"상태: {data['vlm_labels']['status']}")
    for i, cap in enumerate(data['vlm_labels']['captions'], 1):
        print(f"  {i}. {cap}")


파일: 00131_H_A_SY_C3_I001.json
상태: resting
  1. 환자는 현재 침대 위에 편안하게 앉아 휴식 중이며, 특이사항 없이 안정적인 상태를 유지하고 있습니다.
  2. 현재 환자는 침상에 앉아 정적인 자세로 쉬고 있으며, 침대 이탈 징후는 관찰되지 않습니다.
  3. 환자는 침대 중앙에 다리를 모아 앉아있으며, 상체는 곧게 유지된 안정적인 자세로 관찰됩니다.
  4. 낙상 또는 침대 이탈과 관련된 즉각적인 위험은 현재 없으며, 환자는 안전하게 침상에 머무르고 있습니다.
  5. 환자는 침대 위에서 안정적으로 휴식 중입니다.

파일: 00131_H_A_SY_C3_I002.json
상태: bed_exit
  1. 환자가 침대 가장자리에 앉아 양발을 바닥으로 내리는 동작을 보이며 침대 이탈을 시도하고 있습니다.
  2. 현재 환자는 침대 밖으로 나오기 위해 몸을 움직이며 다리를 침대 아래로 내린 상태가 관찰됩니다.
  3. 환자가 침대 끝에 걸터앉아 두 발을 공중에 둔 채 침대에서 내려오려는 자세를 취하고 있습니다.
  4. 침대에서 이탈하려는 환자의 모습이 포착되어 낙상 위험이 있으므로 즉각적인 주의와 개입이 필요합니다.
  5. 환자가 침대에서 이탈하려는 동작이 감지되었습니다.

파일: 00131_H_A_SY_C3_I003.json
상태: bed_exit
  1. 환자가 침대 가장자리에 앉아 목발을 잡고 있으며, 침대에서 벗어나려는 동작을 준비하는 상태가 관찰됩니다.
  2. 현재 환자는 침대 끝에 걸터앉아 하지를 내리고 있으며, 목발을 짚어 침대 이탈을 시도하는 상황입니다.
  3. 환자가 침상 가장자리에 앉아 발을 바닥 쪽으로 내린 채 목발을 잡고 있어 침상 이탈 준비 자세입니다.
  4. 침대 이탈을 시도하는 환자의 움직임이 감지되어 낙상 위험이 증가하므로 즉각적인 확인과 지원이 필요합니다.
  5. 환자가 침대 끝에 앉아 목발을 잡고 침대 밖으로 내려오려는 동작 중입니다.

파일: 00131_H_A_SY_C3_I004.json

---

- Gemini 2.5 Flash 라벨링 시작

In [30]:
import time

# 경로 설정
IMG_SRC = r"C:\Users\ASUS\Desktop\제로베이스\딥러닝 프로젝트\낙상사고 위험동작 영상-센서 쌍 데이터_병원,후면낙상\3.개방데이터\1.데이터\Training\01.원천데이터\TS\이미지"
PROJECT = r"C:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM"
LBL_SRC = os.path.join(PROJECT, "labels", "이미지")

# 구글 드라이브와 로컬에 저장
SAVE_DRIVE = r"G:\내 드라이브\fall_dataset\labels_vlm" # 구글 드라이브
SAVE_LOCAL = os.path.join(PROJECT, "labels_vlm") # 로컬
os.makedirs(SAVE_DRIVE, exist_ok=True)
os.makedirs(SAVE_LOCAL, exist_ok=True)

# 측면낙상(SY) + C3구도만 수집
sy_img_dir = os.path.join(IMG_SRC, "Y", "SY")
sy_lbl_dir = os.path.join(LBL_SRC, "Y", "SY")

scenarios = sorted([s for s in os.listdir(sy_img_dir) if "_C3" in s and os.path.isdir(os.path.join(sy_img_dir, s))])

print(f"SY+C3 시나리오: {len(scenarios)}개, 예상 이미지: {len(scenarios)*10}장")

SY+C3 시나리오: 232개, 예상 이미지: 2320장


In [34]:
# 전체 라벨링 진행
total_done = 0
total_fail = 0
total_skip = 0
start_time = time.time()
for s_idx, scenario in enumerate(scenarios):
    img_folder = os.path.join(sy_img_dir, scenario)
    lbl_folder = os.path.join(sy_lbl_dir, scenario)
    images = sorted([f for f in os.listdir(img_folder) if f.lower().endswith('.jpg')])
    
    print(f"\n[{s_idx+1}/{len(scenarios)}] {scenario} ({len(images)}장)")
    
    for img_name in images:
        json_name = img_name.replace(".jpg", ".json").replace(".JPG", ".json")
        
        # 이미 처리된 파일 건너뛰기 (로컬 기준으로 체크)
        if os.path.exists(os.path.join(SAVE_LOCAL, json_name)):
            total_skip += 1
            continue
        
        # 기존 라벨 읽기
        lbl_path = os.path.join(lbl_folder, json_name)
        if not os.path.exists(lbl_path):
            print(f"  [SKIP] 라벨 없음: {json_name}")
            continue
        
        with open(lbl_path, 'r', encoding='utf-8') as f:
            original_data = json.load(f)
        
        # API 호출
        try:
            vlm_result = call_gemini(os.path.join(img_folder, img_name), PROMPT)
            
            if vlm_result is None:
                total_fail += 1
                print(f"  [FAIL] {img_name}")
                time.sleep(5)
                continue
            
            original_data["vlm_labels"] = {
                "status": vlm_result["status"],
                "captions": vlm_result["captions"]
            }
            
            # 로컬 + Google Drive 둘 다 저장
            for save_dir in [SAVE_LOCAL, SAVE_DRIVE]:
                with open(os.path.join(save_dir, json_name), 'w', encoding='utf-8') as f:
                    json.dump(original_data, f, ensure_ascii=False, indent=2)
            
            total_done += 1
            
        except Exception as e:
            total_fail += 1
            print(f"  [ERROR] {img_name}: {e}")
            time.sleep(5)
        
        time.sleep(4)  # 분당 15회 제한 대비

    # 진행률
    elapsed = time.time() - start_time
    rate = total_done / elapsed * 3600 if elapsed > 0 else 0
    remaining = (len(scenarios)*10 - total_done - total_skip) / (rate/3600) if rate > 0 else 0
    print(f"  ✅ 완료 {total_done} | 건너뜀 {total_skip} | 실패 {total_fail} | {rate:.0f}장/시간 | 남은 ~{remaining/60:.0f}분")
    
print(f"\n{'='*50}")
print(f"전체 완료! 성공: {total_done}, 건너뜀: {total_skip}, 실패: {total_fail}")
print(f"소요 시간: {(time.time()-start_time)/3600:.1f}시간")
print(f"로컬: {SAVE_LOCAL}")
print(f"드라이브: {SAVE_DRIVE}")


[1/232] 00004_H_A_SY_C3 (10장)
  ✅ 완료 10 | 건너뜀 0 | 실패 0 | 209장/시간 | 남은 ~663분

[2/232] 00008_H_A_SY_C3 (10장)
  ✅ 완료 20 | 건너뜀 0 | 실패 0 | 213장/시간 | 남은 ~648분

[3/232] 00012_H_A_SY_C3 (10장)
  ✅ 완료 30 | 건너뜀 0 | 실패 0 | 211장/시간 | 남은 ~652분

[4/232] 00018_H_A_SY_C3 (10장)
  ✅ 완료 40 | 건너뜀 0 | 실패 0 | 210장/시간 | 남은 ~651분

[5/232] 00022_H_A_SY_C3 (10장)
  ✅ 완료 50 | 건너뜀 0 | 실패 0 | 210장/시간 | 남은 ~648분

[6/232] 00034_H_A_SY_C3 (10장)
  ✅ 완료 60 | 건너뜀 0 | 실패 0 | 210장/시간 | 남은 ~646분

[7/232] 00038_H_A_SY_C3 (10장)
  ✅ 완료 70 | 건너뜀 0 | 실패 0 | 211장/시간 | 남은 ~641분

[8/232] 00045_H_A_SY_C3 (10장)
  ✅ 완료 80 | 건너뜀 0 | 실패 0 | 210장/시간 | 남은 ~639분

[9/232] 00049_H_A_SY_C3 (10장)
  ✅ 완료 90 | 건너뜀 0 | 실패 0 | 210장/시간 | 남은 ~636분

[10/232] 00056_H_A_SY_C3 (10장)
  ✅ 완료 100 | 건너뜀 0 | 실패 0 | 211장/시간 | 남은 ~631분

[11/232] 00060_H_A_SY_C3 (10장)
  ✅ 완료 110 | 건너뜀 0 | 실패 0 | 211장/시간 | 남은 ~628분

[12/232] 00064_H_A_SY_C3 (10장)
  ✅ 완료 120 | 건너뜀 0 | 실패 0 | 209장/시간 | 남은 ~631분

[13/232] 00067_H_A_SY_C3 (10장)
  ✅ 완료 130 | 건너뜀 0 | 실패 0 | 209장/시간 | 

In [35]:
# 전체 라벨링 진행
total_done = 0
total_fail = 0
total_skip = 0
start_time = time.time()
for s_idx, scenario in enumerate(scenarios):
    img_folder = os.path.join(sy_img_dir, scenario)
    lbl_folder = os.path.join(sy_lbl_dir, scenario)
    images = sorted([f for f in os.listdir(img_folder) if f.lower().endswith('.jpg')])
    
    print(f"\n[{s_idx+1}/{len(scenarios)}] {scenario} ({len(images)}장)")
    
    for img_name in images:
        json_name = img_name.replace(".jpg", ".json").replace(".JPG", ".json")
        
        # 이미 처리된 파일 건너뛰기 (로컬 기준으로 체크)
        if os.path.exists(os.path.join(SAVE_LOCAL, json_name)):
            total_skip += 1
            continue
        
        # 기존 라벨 읽기
        lbl_path = os.path.join(lbl_folder, json_name)
        if not os.path.exists(lbl_path):
            print(f"  [SKIP] 라벨 없음: {json_name}")
            continue
        
        with open(lbl_path, 'r', encoding='utf-8') as f:
            original_data = json.load(f)
        
        # API 호출
        try:
            vlm_result = call_gemini(os.path.join(img_folder, img_name), PROMPT)
            
            if vlm_result is None:
                total_fail += 1
                print(f"  [FAIL] {img_name}")
                time.sleep(5)
                continue
            
            original_data["vlm_labels"] = {
                "status": vlm_result["status"],
                "captions": vlm_result["captions"]
            }
            
            # 로컬 + Google Drive 둘 다 저장
            for save_dir in [SAVE_LOCAL, SAVE_DRIVE]:
                with open(os.path.join(save_dir, json_name), 'w', encoding='utf-8') as f:
                    json.dump(original_data, f, ensure_ascii=False, indent=2)
            
            total_done += 1
            
        except Exception as e:
            total_fail += 1
            print(f"  [ERROR] {img_name}: {e}")
            time.sleep(5)
        
        time.sleep(4)  # 분당 15회 제한 대비

    # 진행률
    elapsed = time.time() - start_time
    rate = total_done / elapsed * 3600 if elapsed > 0 else 0
    remaining = (len(scenarios)*10 - total_done - total_skip) / (rate/3600) if rate > 0 else 0
    print(f"  ✅ 완료 {total_done} | 건너뜀 {total_skip} | 실패 {total_fail} | {rate:.0f}장/시간 | 남은 ~{remaining/60:.0f}분")
    
print(f"\n{'='*50}")
print(f"전체 완료! 성공: {total_done}, 건너뜀: {total_skip}, 실패: {total_fail}")
print(f"소요 시간: {(time.time()-start_time)/3600:.1f}시간")
print(f"로컬: {SAVE_LOCAL}")
print(f"드라이브: {SAVE_DRIVE}")


[1/232] 00004_H_A_SY_C3 (10장)
  ✅ 완료 0 | 건너뜀 10 | 실패 0 | 0장/시간 | 남은 ~0분

[2/232] 00008_H_A_SY_C3 (10장)
  ✅ 완료 0 | 건너뜀 20 | 실패 0 | 0장/시간 | 남은 ~0분

[3/232] 00012_H_A_SY_C3 (10장)
  ✅ 완료 0 | 건너뜀 30 | 실패 0 | 0장/시간 | 남은 ~0분

[4/232] 00018_H_A_SY_C3 (10장)
  ✅ 완료 0 | 건너뜀 40 | 실패 0 | 0장/시간 | 남은 ~0분

[5/232] 00022_H_A_SY_C3 (10장)
  ✅ 완료 0 | 건너뜀 50 | 실패 0 | 0장/시간 | 남은 ~0분

[6/232] 00034_H_A_SY_C3 (10장)
  ✅ 완료 0 | 건너뜀 60 | 실패 0 | 0장/시간 | 남은 ~0분

[7/232] 00038_H_A_SY_C3 (10장)
  ✅ 완료 0 | 건너뜀 70 | 실패 0 | 0장/시간 | 남은 ~0분

[8/232] 00045_H_A_SY_C3 (10장)
  ✅ 완료 0 | 건너뜀 80 | 실패 0 | 0장/시간 | 남은 ~0분

[9/232] 00049_H_A_SY_C3 (10장)
  ✅ 완료 0 | 건너뜀 90 | 실패 0 | 0장/시간 | 남은 ~0분

[10/232] 00056_H_A_SY_C3 (10장)
  ✅ 완료 0 | 건너뜀 100 | 실패 0 | 0장/시간 | 남은 ~0분

[11/232] 00060_H_A_SY_C3 (10장)
  ✅ 완료 0 | 건너뜀 110 | 실패 0 | 0장/시간 | 남은 ~0분

[12/232] 00064_H_A_SY_C3 (10장)
  ✅ 완료 0 | 건너뜀 120 | 실패 0 | 0장/시간 | 남은 ~0분

[13/232] 00067_H_A_SY_C3 (10장)
  ✅ 완료 0 | 건너뜀 130 | 실패 0 | 0장/시간 | 남은 ~0분

[14/232] 00072_H_A_SY_C3 (10장)
  ✅ 완료 0 | 